In [3]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day05" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "day05"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


* 가드 함수 검사

In [17]:
import importlib
importlib.invalidate_caches() 

from app.core.exceptions import GuardTripped, RateLimited    
from app.core.guards import ALLOWED_MODELS, check_daily_limit, check_model, check_question
from app.core.config import get_settings

settings = get_settings()

def try_guard(label: str, func, arg) -> None:
    try:
        result = func(arg)
    except (GuardTripped, RateLimited) as e:
        code = getattr(e, "status_code", None)
        print(f"{label} : {type(e).__name__} ({code}) {e}")
    else:
        print(f'{label} : 통과 → "{result}"')


print("허용 모델      :", ALLOWED_MODELS)     
print()

try_guard("정상 질문     ", check_question, "부산 출장 숙박비 한도가 얼마인가요?")
try_guard("공백 세 칸    ", check_question, "   ")
try_guard("아주 긴 질문  ", check_question, "가" * (settings.max_input_chars + 1))
try_guard("허용 밖 모델  ", check_model, "claude-opus-4-1")
try_guard("한도 초과     ", check_daily_limit, settings.daily_call_limit)

허용 모델      : {'claude-haiku-4-5'}

정상 질문      : 통과 → "부산 출장 숙박비 한도가 얼마인가요?"
공백 세 칸     : GuardTripped (400) 질문이 비어 있거나 너무 짧습니다.
아주 긴 질문   : GuardTripped (400) 질문이 너무 깁니다 (2001자).2000자 이내로 줄여주세요.
허용 밖 모델   : GuardTripped (400) 허용되지 않은 모델입니다: claude-opus-4-1
한도 초과      : RateLimited (429) 오늘 호출 한도(200회) 를. 모두 사용했습니다.


* 스키마 검사

In [20]:
from pydantic import ValidationError
from app.schemas.chat import AnswerOut


CASES = [
    ("ⓐ 필드 누락  ", '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'),
    ("ⓑ 타입 어긋남", '{"answer": "1박 70,000원 이내입니다.", "sources": "DOC-HR-014", "enough_evidence": true}'),
    ("ⓒ 잡담이 앞에", '네, 알겠습니다!\n{"answer": "1박 70,000원 이내입니다.", "sources": [], "enough_evidence": false}'),
]

for label, raw in CASES:
    try : 
        AnswerOut.model_validate_json(raw)
        print(f"{label} : 통과")
    except ValidationError as e:
        first =e.errors(include_url=False)[0]
        print(f"{label}: loc={str(first['loc'])}/ type={first['type']} / {first['msg']}")

ⓐ 필드 누락  : loc=('sources',)/ type=missing / Field required
ⓑ 타입 어긋남: loc=('sources',)/ type=list_type / Input should be a valid array
ⓒ 잡담이 앞에: loc=()/ type=json_invalid / Invalid JSON: expected value at line 1 column 1


* 재시도, 풀백 검증 함수

In [22]:
import importlib
import app.schemas.chat as chat_schema

chat_schema = importlib.reload(chat_schema)

print('AskOut 필드 :', list(chat_schema.AskOut.model_fields))

sample = chat_schema.AskOut(run_id='RUN-8821', answer='1박 70,000원 이내입니다.', sources=[], enough_evidence=False)
print(f'기본값      : attempts={sample.attempts} · fallback_used={sample.fallback_used}')

AskOut 필드 : ['answer', 'sources', 'enough_evidence', 'run_id', 'attempts', 'fallback_used']
기본값      : attempts=1 · fallback_used=False


* fallback 테스트

In [23]:
import importlib
import inspect

importlib.invalidate_caches()    

from app.services import chat_service

print("ask 시그니처 :", inspect.signature(chat_service.ask, eval_str=True))
print(f"MAX_ATTEMPTS : {chat_service.MAX_ATTEMPTS}   (첫 호출 1 + 재시도 2)")


ask_body = inspect.getsource(chat_service.ask)

caught_names = []                                 
for line in ask_body.splitlines():
    stripped = line.strip()                     
    if not stripped.startswith("except "):
        continue                                

    after_except = stripped[len("except "):]      
    exception_name = after_except.split(" as ")[0]  
    caught_names.append(exception_name.rstrip(":")) 

if len(caught_names) == 1:
    how_many = "하나"
else:
    how_many = f"{len(caught_names)}개"
print("잡는 예외    :", " · ".join(caught_names), how_many)

ask 시그니처 : (*, question: str, run_id: str = 'RUN-0000') -> app.schemas.chat.AskOut
MAX_ATTEMPTS : 3   (첫 호출 1 + 재시도 2)
잡는 예외    : ValidationError 하나


In [24]:
import json
# 잘된 응답 
GOOD = json.dumps(
    {
        "answer": "국내출장 여비 규정 제12조에 따르면 숙박비는 1박 70,000원 이내입니다.",
        "sources": [
            {
                "doc_id": "DOC-HR-014",
                "title": "국내출장 여비 규정",
                "version": "v2.0",
                "locator": "제12조",
            }
        ],
        "enough_evidence": True,
    },
    ensure_ascii=False,
)
# 잘못된 응답 
NO_SOURCES = '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'
CHITCHAT = '죄송합니다. 지금은 답변을 드릴 수 없습니다.'

def next_answer(replies: list[str], attempt: int) -> str:
    if attempt <= len(replies):
        return replies[attempt - 1]
    return replies[-1]
print('준비한 응답 :', 'GOOD · NO_SOURCES · CHITCHAT 세 벌')


준비한 응답 : GOOD · NO_SOURCES · CHITCHAT 세 벌


In [25]:
from pydantic import ValidationError
from app.integrations.llm_claude import _extract_json
from app.schemas.chat import AnswerOut, AskOut
from app.services.chat_service import MAX_ATTEMPTS, _fallback, _hint_from

def run_loop(replies: list[str], run_id: str='RUN-0000') -> AskOut:
    hint = ''
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw = next_answer(replies, attempt)
        try:
            data = _extract_json(raw)
            AnswerOut.model_validate(data)
        except ValidationError as exc:
            hint = _hint_from(exc.errors(include_url=False))
            continue
        return AskOut(**data, run_id=run_id, attempts=attempt, fallback_used=False)
    return _fallback(run_id, MAX_ATTEMPTS)
CASES_RETRY = [('① 한 번에 성공  ', [GOOD]), ('② 한 번 실패    ', [NO_SOURCES, GOOD]), ('③ 전부 실패     ', [CHITCHAT, CHITCHAT, CHITCHAT])]
for (label, replies) in CASES_RETRY:
    out = run_loop(replies)
    source_count = len(out.sources)
    print(f'{label}{out.run_id}  attempts={out.attempts}  fallback={str(out.fallback_used):<6} 근거 {source_count}건')
    if out.fallback_used:
        print(f'   ⚠️ 폴백으로 응답합니다 (attempts={out.attempts})')


① 한 번에 성공  RUN-0000  attempts=1  fallback=False  근거 1건
② 한 번 실패    RUN-0000  attempts=2  fallback=False  근거 1건
③ 전부 실패     RUN-0000  attempts=3  fallback=True   근거 0건
   ⚠️ 폴백으로 응답합니다 (attempts=3)
